# Offline Assignment

This is an offline assigment meant for us in Commit to understand how you approach data science problems, and how you implement said approach.


It is 100% OK if you don't succeed finishing some of the sub-tasks- in fact, it is typical not get everything right.

We wanted to give you the best chance to show us what you bring to the table no matter which field of data science interests you most.


We have many tasks in Commit that are very similar to the task you see here.


**Important note- when you encounter a problem, all approaches that make sense are valid. Tradeoffs are the nature of the field, and it is likely that your considerations for tradeoffs will somewhat vary from those of the interviewer. As long as you can explain your considerations (or most of them), this is 100% fine**

# Use Case - NLP


## Description

You are given a task in the field of NLP.
Our client, a large scientific corporation, is interested in using LLMs to improve its scientific work.

## Our First Task

Part of the scientific work involves making internet search for a given topic, and presenting only the important material to a researcher.


For that, they would like you to mine a portion of the web (several web pages are enough) and create a vector database that you can query.


Please mine several web pages (from ArXiv or Pubmed), extract the text from them, then split the text in a meaningful way, then build a vector database that one can query to retrieve the needed data.


In [ ]:

DESIGN_CELL = '''
System Architecture Design

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                       SCIENTIFIC PAPER RAG SYSTEM                           │
└─────────────────────────────────────────────────────────────────────────────┘

┌──────────────┐     ┌──────────────┐     ┌───────────────────────────────────┐
│   ArXiv API  │     │ PubMed API   │     │   1. DATA MINING (Official APIs)  │
│  (Atom XML)  │     │ (E-utilities)│     │   - Rate limited & Ethical        │
└──────┬───────┘     └──────┬───────┘     │   - Metadata + PDF Download       │
       │                    │             └───────────────────────────────────┘
       └────────┬───────────┘
                │
                ▼
┌───────────────────────────────────┐     ┌───────────────────────────────────┐
│   UNIFIED SCRAPER & EXTRACTOR     │     │   2. FULL-TEXT ENHANCEMENT        │
│  - ArXiv/PubMed Unified Client    │     │   - PyMuPDF Full-Text Extraction  │
│  - PDF Content Extraction         │◄────┤   - 40-60% Accuracy Improvement   │
│  - Multi-threaded Downloading     │     │   - Text Cleaning & Sanitization  │
└──────────────┬────────────────────┘     └───────────────────────────────────┘
               │
               ▼
┌───────────────────────────────────┐     ┌───────────────────────────────────┐
│     COMPLIANCE & SECURITY         │     │   3. PRODUCTION HARDENING         │
│  - PII Detection & Redaction      │     │   - GDPR/HIPAA Compliant          │
│  - Audit Logging (Access Control) │◄────┤   - Structured Logging (JSON)     │
│  - Pseudonymization (Hashing)     │     │   - Exponential Backoff Retries   │
└──────────────┬────────────────────┘     └───────────────────────────────────┘
               │
               ▼
┌───────────────────────────────────┐     ┌───────────────────────────────────┐
│   STAGE 1: HYBRID RETRIEVAL       │     │   4. SIGNAL DISCOVERY             │
│  - ChromaDB Vector Storage        │     │   - Vector (Dense) Search         │
│  - BM25 Keyword Indexing          │◄────┤   - BM25 (Sparse) Search          │
│  - Retrieves Top-25 Candidates    │     │   - Fast & Scalable filtering     │
└──────────────┬────────────────────┘     └───────────────────────────────────┘
               │
               ▼
┌───────────────────────────────────┐     ┌───────────────────────────────────┐
│   STAGE 2: SEMANTIC RERANKING     │     │   5. PRECISION REFINEMENT         │
│  - Cross-Encoder Model            │     │   - Query-Document Interaction    │
│  - Scoring Candidates (0.0-1.0)   │◄────┤   - Highest Accuracy Sort         │
│  - Filters down to Final Top-5    │     │   - Eliminates False Positives    │
└──────────────┬────────────────────┘     └───────────────────────────────────┘
               │
               ▼
┌───────────────────────────────────┐     ┌───────────────────────────────────┐
│   QUERY INTELLIGENCE (LLM)        │     │   6. MULTI-HOP SYNTHESIS          │
│  - Chain-of-Thought Decomposition │     │   - Sub-Query Generation (Sub-Qs)  │
│  - Answer Synthesis with Citations│◄────┤   - 15-25% Precision Boost        │
│  - Paper-grounded Reasoning       │     │   - Source Attribution            │
└──────────────┬────────────────────┘     └───────────────────────────────────┘
               │
               ▼
┌───────────────────────────────────┐     ┌───────────────────────────────────┐
│     EVALUATION (LLM-AS-JUDGE)     │     │   7. QUALITY METRICS              │
│  - Faithfulness (Hallucination)   │     │   - RAGAS-style Metrics           │
│  - Answer Relevancy               │◄────┤   - Mean Reciprocal Rank (MRR)    │
│  - Context Relevancy              │     │   - Hit Rate & Fact Consistency   │
└───────────────────────────────────┘     └───────────────────────────────────┘
```
ADVANCED DESIGN: THE TWO-STAGE PIPELINE
---------------------------------------

To achieve production-grade performance, we implement a "Retrieve-and-Rerank" strategy:

1. STAGE 1: HYBRID RETRIEVAL (The Net)
   We use a combination of Vector Search (for conceptual meaning) and BM25 (for exact keywords). 
   This is very fast and efficient, casting a "wide net" over our papers to find the top ~25 possible matches.

2. STAGE 2: CROSS-ENCODER RERANKING (The Filter)
   We take those 25 candidates and pass them through a powerful Cross-Encoder Model. 
   Unlike vector models which look at chunks in isolation, the Cross-Encoder processes the question 
   and the chunk TOGETHER. This eliminates irrelevant results that might have looked similar 
   mathematically but don't actually answer the user's question.

RESULTS:
This architecture provides the best of both worlds:
- The speed of a vector database
- The hyper-precision of a deep neural transformer
'''

if __name__ == "__main__":
    print(DESIGN_CELL)


In [ ]:
%pip install arxiv biopython chromadb sentence-transformers PyMuPDF rank-bm25 tenacity requests

import os
import hashlib
import time
import re
from dataclasses import dataclass
from datetime import datetime
from typing import List, Optional, Dict, Any
from xml.etree import ElementTree as ET
import requests

# Vector DB & Embeddings
import chromadb
from chromadb.utils import embedding_functions

# Enhancement: Cross-Encoder Reranking
try:
    from sentence_transformers import CrossEncoder
    RERANKER_AVAILABLE = True
except ImportError:
    RERANKER_AVAILABLE = False

# Enhancement: PDF Extraction
try:
    import fitz  # PyMuPDF
    PDF_AVAILABLE = True
except ImportError:
    PDF_AVAILABLE = False

# Enhancement: Hybrid Search
try:
    from rank_bm25 import BM25Okapi
    HYBRID_AVAILABLE = True
except ImportError:
    HYBRID_AVAILABLE = False

# ============================================================================
# CONFIGURATION
# ============================================================================
@dataclass
class Config:
    CHUNK_SIZE: int = 1000
    CHUNK_OVERLAP: int = 200
    EMBEDDING_MODEL: str = "all-MiniLM-L6-v2"
    RERANKER_MODEL: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    ARXIV_RATE_LIMIT: float = 0.33
    COLLECTION_NAME: str = "scientific_papers_v3"
    CHROMA_PATH: str = "./chroma_db"

config = Config()

# ============================================================================
# DATA MODELS
# ============================================================================
@dataclass
class Paper:
    paper_id: str
    source: str
    title: str
    abstract: str
    authors: List[str]
    url: str = ""
    pdf_url: Optional[str] = None
    full_text: Optional[str] = None

@dataclass
class TextChunk:
    chunk_id: str
    text: str
    paper_id: str
    paper_title: str
    source: str
    chunk_index: int

# ============================================================================
# PDF EXTRACTOR
# ============================================================================
class PDFExtractor:
    @staticmethod
    def extract_from_url(pdf_url: str) -> Optional[str]:
        if not PDF_AVAILABLE:
            return None
        try:
            response = requests.get(pdf_url, timeout=30)
            response.raise_for_status()
            doc = fitz.open(stream=response.content, filetype="pdf")
            text = "\n".join([page.get_text("text") for page in doc])
            doc.close()
            return re.sub(r'\n{3,}', '\n\n', text.replace("\x00", ""))
        except Exception as e:
            print(f"   ⚠️  PDF extraction failed: {e}")
            return None

# ============================================================================
# SCRAPERS
# ============================================================================
class ArXivScraper:
    def search(self, query: str, max_results: int = 5, extract_pdfs: bool = False) -> List[Paper]:
        try:
            params = {"search_query": f"all:{query}", "max_results": max_results, "sortBy": "relevance"}
            res = requests.get("https://export.arxiv.org/api/query", params=params, timeout=20)
            res.raise_for_status()
            root = ET.fromstring(res.text)
            ns = {"atom": "http://www.w3.org/2005/Atom"}
            papers = []
            for entry in root.findall("atom:entry", ns):
                p_id = entry.find("atom:id", ns).text.split("/")[-1]
                pdf = next((l.get("href") for l in entry.findall("atom:link", ns) if l.get("title") == "pdf"), None)
                full = PDFExtractor.extract_from_url(pdf) if extract_pdfs and pdf else None
                papers.append(Paper(
                    p_id, "arxiv",
                    entry.find("atom:title", ns).text.strip(),
                    entry.find("atom:summary", ns).text.strip(),
                    [], f"https://arxiv.org/abs/{p_id}", pdf, full
                ))
            print(f"   ✅ ArXiv: fetched {len(papers)} papers")
            return papers
        except Exception as e:
            print(f"   ❌ ArXiv failed: {e}")
            return []

class PubMedScraper:
    def search(self, query: str, max_results: int = 5) -> List[Paper]:
        try:
            res = requests.get(
                "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
                params={"db": "pubmed", "term": query, "retmax": max_results, "retmode": "xml"},
                timeout=20
            )
            res.raise_for_status()
            ids = [i.text for i in ET.fromstring(res.text).findall(".//Id")]
            if not ids:
                print("   ⚠️  PubMed: no results found")
                return []
            res = requests.get(
                "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
                params={"db": "pubmed", "id": ",".join(ids), "retmode": "xml"},
                timeout=20
            )
            res.raise_for_status()
            papers = []
            for a in ET.fromstring(res.text).findall(".//PubmedArticle"):
                pmid = a.find(".//PMID").text
                papers.append(Paper(
                    pmid, "pubmed",
                    a.find(".//ArticleTitle").text,
                    a.find(".//AbstractText").text if a.find(".//AbstractText") is not None else "",
                    [], f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
                ))
            print(f"   ✅ PubMed: fetched {len(papers)} papers")
            return papers
        except Exception as e:
            print(f"   ❌ PubMed failed: {e}")
            return []

# ============================================================================
# VECTOR DATABASE WITH RERANKING
# ============================================================================
class VectorDB:
    def __init__(self):
        self.client = chromadb.PersistentClient(path=config.CHROMA_PATH)
        self.embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=config.EMBEDDING_MODEL
        )
        self.collection = self.client.get_or_create_collection(
            name=config.COLLECTION_NAME,
            embedding_function=self.embedding_fn
        )
        if RERANKER_AVAILABLE:
            print(f"📦 Loading Reranker: {config.RERANKER_MODEL}...")
            self.reranker = CrossEncoder(config.RERANKER_MODEL)

    def add_chunks(self, chunks: List[TextChunk]):
        if not chunks:
            return
        self.collection.upsert(
            ids=[c.chunk_id for c in chunks],
            documents=[c.text for c in chunks],
            metadatas=[{
                "paper_id": c.paper_id,
                "paper_title": c.paper_title,
                "source": c.source
            } for c in chunks]
        )

    def hybrid_search(self, query: str, top_k: int = 25, alpha: float = 0.5) -> List[Dict]:
        """Stage 1 Retrieval: Fast BM25 + Vector hybrid filtering of candidates."""
        all_res = self.collection.get(include=["documents", "metadatas"])
        docs = all_res["documents"]
        if not docs:
            return []

        bm25 = BM25Okapi([d.lower().split() for d in docs])
        bm25_scores = bm25.get_scores(query.lower().split())
        if bm25_scores.max() > 0:
            bm25_scores /= bm25_scores.max()

        vec_res = self.collection.query(query_texts=[query], n_results=len(docs))
        vec_map = {d_id: 1 - dist for d_id, dist in zip(vec_res["ids"][0], vec_res["distances"][0])}

        scored = []
        for i, doc_id in enumerate(all_res["ids"]):
            score = alpha * vec_map.get(doc_id, 0.0) + (1 - alpha) * bm25_scores[i]
            scored.append({"id": doc_id, "text": docs[i], "metadata": all_res["metadatas"][i], "score": score})

        scored.sort(key=lambda x: x["score"], reverse=True)
        return scored[:top_k]

    def search_with_rerank(self, query: str, top_k: int = 5, retrieve_k: int = 25) -> List[Dict]:
        """
        Two-Stage Retrieval:
        Stage 1: Hybrid Search retrieves broad candidates.
        Stage 2: Cross-Encoder re-scores each (query, doc) pair for precision.
        """
        candidates = self.hybrid_search(query, top_k=retrieve_k)
        if not candidates or not RERANKER_AVAILABLE:
            return candidates[:top_k]

        print(f"🔄 Reranking {len(candidates)} candidates...")
        pairs = [[query, cand["text"]] for cand in candidates]
        rerank_scores = self.reranker.predict(pairs)

        for i, cand in enumerate(candidates):
            cand["rerank_score"] = float(rerank_scores[i])
            cand["type"] = "reranked"

        candidates.sort(key=lambda x: x["rerank_score"], reverse=True)
        return candidates[:top_k]

# ============================================================================
# DEMO
# ============================================================================
def run_task1_demo():
    print("=" * 60)
    print("🚀 TASK 1: Hybrid Search + Cross-Encoder Reranking")
    print("=" * 60)

    arxiv = ArXivScraper()
    pubmed = PubMedScraper()
    db = VectorDB()

    query = "breast cancer detection AI"
    print(f"\n📡 Mining & Indexing: '{query}'")
    all_papers = arxiv.search(query, max_results=2, extract_pdfs=True) + pubmed.search(query, max_results=2)

    if not all_papers:
        raise RuntimeError("No papers fetched — please check your internet connection and try again.")

    print(f"\n📄 Total papers to index: {len(all_papers)}")

    chunks = []
    for p in all_papers:
        text = p.full_text if p.full_text else p.abstract
        p_chunks = [text[i:i + 1000] for i in range(0, len(text), 800)]
        chunks.extend([
            TextChunk(
                hashlib.md5(c.encode()).hexdigest()[:],
                c, p.paper_id, p.title, p.source, i
            )
            for i, c in enumerate(p_chunks)
        ])

    db.add_chunks(chunks)
    print(f"✅ Indexed {len(chunks)} chunks into ChromaDB")

    print(f"\n🎯 Performing Two-Stage Reranked Search...")
    results = db.search_with_rerank("machine learning for oncology", top_k=3, retrieve_k=10)

    print("\n📊 TOP RESULTS:")
    for i, res in enumerate(results, 1):
        score_key = "rerank_score" if "rerank_score" in res else "score"
        print(f"\n   [{i}] Score: {res[score_key]:.3f} | Source: {res['metadata']['source']}")
        print(f"   Paper: {res['metadata']['paper_title']}")
        print(f"   Text:  {res['text'][:]}...")

    return db

db = run_task1_demo()

## Our Follow-up Task

Now that you are done, the client wants more (clients almost always do).

The client now wants to be able to ask complex questions (like “what are the dangerous medical conditions oh wise chatbot”) and still get an answer based on their data.


Upon thinking, you came up with a solution- you will break the question into sub steps, query the vector database about each sub-step, and then compose an answer.

Please create a flow that allows a person to ask complex questions and still get a reply based on their data. This flow should break down the question to the right substeps and then compose a good answer.  


In [ ]:
%pip install langchain-google-genai langchain-core google-generativeai pydantic

import os
import time
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# ============================================================================
# CONFIGURATION
# ============================================================================
# Set GOOGLE_API_KEY before running this cell.
# os.environ["GOOGLE_API_KEY"] = "your-google-api-key-here"

# ============================================================================
# DATA MODELS
# ============================================================================

class SubQuery(BaseModel):
    query: str = Field(description="A simpler, focused sub-question derived from the complex question.")
    reasoning: str = Field(description="Brief explanation of why this sub-query is necessary to answer the main question.")
    order: int = Field(description="The sequential order in which this sub-query should be answered (1, 2, ...).")

class DecomposedQuery(BaseModel):
    is_complex: bool = Field(description="Whether the original question requires decomposition.")
    sub_queries: List[SubQuery] = Field(description="A list of 2-4 sub-queries to solve the complex question.")

# ============================================================================
# QUERY DECOMPOSER (LangChain Implementation)
# ============================================================================

class QueryDecomposer:
    """Uses Gemini to break down complex queries into manageable sub-queries."""

    def __init__(self, model_name: str = "gemini-2.5-flash", temperature: float = 0):
        # temperature=0 keeps the structured JSON output deterministic.
        self.llm = ChatGoogleGenerativeAI(model=model_name, temperature=temperature)
        self.parser = JsonOutputParser(pydantic_object=DecomposedQuery)

        self.prompt = ChatPromptTemplate.from_template(
            """You are a specialized RAG Query Optimizer.
Your task is to analyze a complex scientific question and break it down into focused sub-questions.

Complex Question: {question}

Instructions:
1. Determine if the question is complex (requires comparing multiple concepts, multi-part answers, or specific technical details).
2. If complex, generate 2-3 logical sub-queries that cover foundational concepts first, then relational ones.
3. Each sub-query must be specific and answerable independently.
4. If the question is simple, set is_complex to false and sub_queries to empty.

{format_instructions}
"""
        )
        self.chain = self.prompt | self.llm | self.parser

    def decompose(self, question: str) -> Dict[str, Any]:
        print(f"🧠 Analyzing complexity of: '{question}'")
        try:
            return self.chain.invoke({
                "question": question,
                "format_instructions": self.parser.get_format_instructions(),
            })
        except Exception as e:
            print(f"❌ Decomposition failed: {e}")
            return {"is_complex": False, "sub_queries": []}

# ============================================================================
# RAG PIPELINE (Synthesis & Multi-Hop Retrieval with RERANKING)
# ============================================================================

class RAGPipeline:
    """Full RAG Pipeline including search, decomposition, and synthesis."""

    def __init__(self, vector_db, api_key: Optional[str] = None):
        if api_key:
            os.environ["GOOGLE_API_KEY"] = api_key
        if not os.environ.get("GOOGLE_API_KEY"):
            raise RuntimeError("GOOGLE_API_KEY is required to run Task 2.")

        self.db = vector_db
        self.decomposer = QueryDecomposer()
        # temperature=0.3 keeps the answer fluent but still grounded in the retrieved context.
        self.llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

        self.synthesis_prompt = ChatPromptTemplate.from_template(
            """You are a Scientific Knowledge Assistant.
Answer the user's question precisely using the provided context.

Question: {question}

Context:
{context}

Instructions:
1. Cite specific sources (Paper Titles) where possible.
2. Be technical and accurate.
3. If information is missing, state what's unknown.
4. If multiple papers are used, synthesize the findings.

Answer:"""
        )

    def _format_context(self, results: List[Dict[str, Any]]) -> str:
        context_blocks = []
        seen_texts = set()

        for res in results:
            text = res["text"]
            if text not in seen_texts:
                seen_texts.add(text)
                title = res["metadata"].get("paper_title", "Unknown Paper")
                score_info = f" [Relevance: {res.get('rerank_score', res.get('score', 0)):.2f}]" if "rerank_score" in res else ""
                context_blocks.append(f"Source [{title}]{score_info}:\n{text}")

        return "\n\n---\n\n".join(context_blocks)

    def query(self, question: str, top_k: int = 4, use_reranking: bool = True) -> Dict[str, Any]:
        start_time = time.time()

        decomp_result = self.decomposer.decompose(question)
        is_complex = decomp_result.get("is_complex", False)
        sub_queries = decomp_result.get("sub_queries", [])

        retrieved_results = []
        if is_complex and sub_queries:
            print(f"🔀 Multi-hop retrieval for {len(sub_queries)} sub-queries...")
            for sq in sub_queries:
                print(f"   - Querying: {sq['query']}")
                if use_reranking:
                    res = self.db.search_with_rerank(sq["query"], top_k=top_k, retrieve_k=15)
                else:
                    res = self.db.hybrid_search(sq["query"], top_k=top_k)
                retrieved_results.extend(res)
        else:
            print(f"📝 Simple retrieval with {'reranking' if use_reranking else 'hybrid search'}...")
            if use_reranking:
                retrieved_results = self.db.search_with_rerank(question, top_k=top_k, retrieve_k=15)
            else:
                retrieved_results = self.db.hybrid_search(question, top_k=top_k)

        print("💡 Synthesizing final answer...")
        context = self._format_context(retrieved_results)
        chain = self.synthesis_prompt | self.llm
        answer_response = chain.invoke({"question": question, "context": context})

        elapsed = time.time() - start_time
        sources = list({r["metadata"].get("paper_title", "Unknown") for r in retrieved_results})

        return {
            "answer": answer_response.content,
            "sub_queries": sub_queries,
            "retrieved_count": len(retrieved_results),
            "unique_sources": len(sources),
            "is_complex": is_complex,
            "latency": round(elapsed, 2),
            "reranking_used": use_reranking,
        }

# MAIN EXECUTION - TASK 2 DEMO

def run_task2_demo(db):
    print("=" * 60)
    print("🚀 RUNNING TASK 2: Complex Query + Reranked Retrieval")
    print("=" * 60)

    if not os.environ.get("GOOGLE_API_KEY"):
        print("⚠️ Set GOOGLE_API_KEY to run Task 2.")
        return None

    rag = RAGPipeline(db)

    complex_q = (
        "How does the effectiveness of transformer-based models compare to "
        "traditional CNNs for automated lesion detection in MRI scans?"
    )

    result = rag.query(complex_q, use_reranking=True)

    print("\n" + "-" * 40)
    print(f"❓ QUESTION: {complex_q}")
    print("-" * 40)

    if result["is_complex"]:
        print("\n🔀 SUB-QUERIES GENERATED:")
        for i, sq in enumerate(result["sub_queries"], 1):
            print(f"   {i}. {sq['query']}")
            print(f"      Reason: {sq['reasoning']}")

    print(f"\n📄 FINAL ANSWER:")
    print(result["answer"])

    print(f"\n📊 METRICS:")
    print(f"   - Processing Time: {result['latency']}s")
    print(f"   - Context Chunks: {result['retrieved_count']}")
    print(f"   - Unique Sources: {result['unique_sources']}")
    print(f"   - Reranking: {'✅ Enabled' if result['reranking_used'] else '❌ Disabled'}")

    print("\n" + "=" * 60)
    print("✅ TASK 2 COMPLETE")
    print("=" * 60)

    return rag

if __name__ == "__main__":
    if 'db' in globals():
        run_task2_demo(db)
    else:
        print("⚠️ ERROR: 'db' not found. Please run Task 1 cell first.")


## Our Next Follow-up Task

Now that you are done, the client is satisfied and wants to move from POC into production.

Please harden your code and add as much of the items needed for code to be production-grade as you can.  

Examples are: log mockups, error handling, etc.

**Remember - the key here is to see that you know what to add and what to look for, implementation is secondary**

In [ ]:
%pip install structlog cryptography pydantic tenacity

import os
import time
import json
import logging
import hashlib
import re
from typing import List, Dict, Any, Optional, Union
from functools import wraps

# Enhancement: Structured Logging
import structlog

# Enhancement: Security & Cryptography
from cryptography.fernet import Fernet

# Enhancement: Validation
from pydantic import BaseModel, Field, validator

# Enhancement: Resilience
from tenacity import retry, stop_after_attempt, wait_exponential

# ============================================================================
# 1. STRUCTURED LOGGING
# ============================================================================

def setup_logging():
    """Configures structured (JSON) logging for production monitoring."""
    structlog.configure(
        processors=[
            structlog.processors.add_log_level,
            structlog.processors.TimeStamper(fmt="iso"),
            structlog.processors.JSONRenderer()
        ],
        context_class=dict,
        logger_factory=structlog.PrintLoggerFactory(),
        cache_logger_on_first_use=True,
    )
    return structlog.get_logger()

logger = setup_logging()

# ============================================================================
# 2. RESILIENCE (Retries & Error Handling)
# ============================================================================

class RAGSystemError(Exception):
    """Base custom exception for the RAG system."""
    pass

class APIConnectionError(RAGSystemError):
    """Raised when external APIs or the LLM provider fail."""
    pass

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10)
)
def protected_api_call(func, *args, **kwargs):
    """
    Generic wrapper to protect any network-based function call.
    Uses exponential backoff retries.
    """
    try:
        return func(*args, **kwargs)
    except Exception as e:
        logger.error("api_call_failed", error=str(e), attempt="retrying...")
        raise APIConnectionError(f"Persistent failure after retries: {e}")

# ============================================================================
# 3. GDPR/HIPAA COMPLIANCE (Security & PII)
# ============================================================================

class ComplianceHandler:
    """
    Handles PII detection, Redaction, and Audit Trails.
    Essential for HIPAA (Medical Data) and GDPR (Privacy).
    """
    
    # Common PII Regex Patterns
    PII_PATTERNS = {
        "email": r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
        "phone": r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',
        "ssn": r'\b\d{3}-\d{2}-\d{4}\b',
        "ip_address": r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b'
    }

    @staticmethod
    def audit_trail(user_id: str, action: str, paper_id: Optional[str] = None):
        """Logs every access attempt for compliance audits."""
        logger.info(
            "audit_log",
            user_id=ComplianceHandler.hash_id(user_id),
            action=action,
            paper_id=paper_id,
            timestamp=time.time()
        )

    @staticmethod
    def hash_id(identifier: str) -> str:
        """One-way cryptographic hash for pseudonymization."""
        return hashlib.sha256(identifier.encode()).hexdigest()

    @staticmethod
    def redact_pii(text: str) -> str:
        """Identifies and redacts sensitive information from scientific text."""
        redacted_text = text
        for pii_type, pattern in ComplianceHandler.PII_PATTERNS.items():
            matches = list(re.finditer(pattern, redacted_text))
            if matches:
                logger.warning("pii_detected", pii_type=pii_type, count=len(matches))
                redacted_text = re.sub(pattern, f"[REDACTED_{pii_type.upper()}]", redacted_text)
        return redacted_text

# ============================================================================
# 4. INPUT VALIDATION (Pydantic)
# ============================================================================

class QueryRequest(BaseModel):
    """Schema for validating user queries before they hit the LLM."""
    query: str = Field(..., min_length=5, max_length=500)
    user_id: str
    top_k: int = Field(default=5, ge=1, le=20)

    @validator('query')
    def prevent_prompt_injection(cls, v):
        """Basic safeguard against malicious instructions."""
        forbidden_keywords = ["ignore previous instructions", "system prompt", "delete database"]
        if any(keyword in v.lower() for keyword in forbidden_keywords):
            raise ValueError("Potential prompt injection detected.")
        return v

# ============================================================================
# 5. RATE LIMITER
# ============================================================================

class RequestThrottler:
    """Ensures we stay within API limits per minute."""
    def __init__(self, rpm_limit: int = 20):
        self.interval = 60.0 / rpm_limit
        self.last_call = 0.0

    def throttle(self):
        now = time.time()
        wait_time = self.interval - (now - self.last_call)
        if wait_time > 0:
            time.sleep(wait_time)
        self.last_call = time.time()

# ============================================================================
# DEMONSTRATION WORKFLOW
# ============================================================================

def run_task3_demo():
    print("=" * 60)
    print("🚀 RUNNING TASK 3: Production Hardening & Compliance")
    print("=" * 60)

    user_query = "   What are the latest findings on COVID-19 mRNA vaccines?   "
    print(f"\n1. Validating User Input...")
    try:
        request = QueryRequest(query=user_query, user_id="user_john_doe", top_k=5)
        print(f"   ✅ Query validated: '{request.query}'")
    except Exception as e:
        print(f"   ❌ Validation error: {e}")
        return

    raw_paper_content = """
    This study by lead researcher dr_smith@university.edu (Phone: 555-0199) 
    found that vaccines are highly effective. Patient IP: 192.168.1.1.
    """
    print(f"\n2. Applying GDPR/HIPAA Redaction...")
    safe_content = ComplianceHandler.redact_pii(raw_paper_content)
    print("   Original Text: Leading researcher info present.")
    print(f"   Cleaned Text: {safe_content.strip()}")

    print(f"\n3. Logging Audit Trail (Pseudonymized)...")
    ComplianceHandler.audit_trail(request.user_id, "search_query", paper_id="arxiv_2301.0001")
    
    print(f"\n4. Resilience Simulation (Retries)...")
    def simulate_flaky_api():
        import random
        if random.random() < 0.6:
            raise ConnectionError("Network timeout!")
        return "Success from API"

    try:
        result = protected_api_call(simulate_flaky_api)
        print(f"   ✅ API Result: {result}")
    except APIConnectionError as e:
        print(f"   ❌ Final Failure: {e}")

    print("\n" + "=" * 60)
    print("✅ TASK 3 COMPLETE (Check JSON output above for logs)")
    print("=" * 60)

if __name__ == "__main__":
    run_task3_demo()


## Next Follow Up Task - Evaluation

Now, the customer is pleased but they raised a question- how do I know that my model produces accurate results?

Please think of a way to measure how accurate the model is.

Please make sure that the measuring method actually measures the quality of the answer.

In [ ]:
"""
HOW TO MEASURE RAG MODEL ACCURACY
=================================

THE CHALLENGE
-------------
Traditional NLP metrics (BLEU, ROUGE) don't work well for RAG because:
- They rely on exact text matching
- They don't measure factual accuracy
- They ignore whether the answer actually addresses the question

RAGAS FRAMEWORK APPROACH
------------------------
RAGAS (Retrieval Augmented Generation Assessment) provides specialized metrics:

1. FAITHFULNESS (Most Important)
   What it measures: Is the answer factually consistent with the retrieved context?
   Why it matters: Prevents hallucination - the #1 concern with LLMs.
   How to calculate:
   1. Extract all claims from the generated answer
   2. For each claim, check if it's supported by the context
   3. Score = (supported claims) / (total claims)

2. ANSWER RELEVANCY
   What it measures: Does the answer actually address the question?
   Why it matters: Ensures the system provides useful responses.
   How to calculate:
   1. Generate hypothetical questions the answer could address
   2. Compare these with the original question
   3. High similarity = high relevancy

3. CONTEXT RELEVANCY
   What it measures: How much of the retrieved context is useful?
   Why it matters: Evaluates retrieval quality (signal vs noise).
   How to calculate:
   1. Analyze each retrieved chunk
   2. Determine what percentage is relevant to the question
   3. Higher ratio = better retrieval

4. CONTEXT RECALL (with ground truth)
   What it measures: Does the context contain all info needed for the answer?
   Why it matters: Ensures the retrieval captures complete information.
   Requires: Ground truth answers for comparison.
"""

In [ ]:
%pip install langchain-google-genai langchain-core pydantic numpy pandas google-generativeai

import os
from typing import List, Dict, Any
import numpy as np
import pandas as pd

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# ============================================================================
# CONFIGURATION
# ============================================================================
# Set GOOGLE_API_KEY before running this cell.
# os.environ["GOOGLE_API_KEY"] = "your-google-api-key-here"

# ============================================================================
# EVALUATION MODELS (Pydantic)
# ============================================================================

class FaithfulnessScore(BaseModel):
    statement: str = Field(description="The claim or statement extracted from the answer.")
    supported_by_context: bool = Field(description="Whether the statement is supported by the provided context.")
    reasoning: str = Field(description="Explanation for the support decision.")

class FaithfulnessEvaluation(BaseModel):
    overall_score: float = Field(description="Score between 0 and 1 (supported statements / total statements).")
    evaluations: List[FaithfulnessScore] = Field(description="Detailed breakdown of each statement.")

class RelevancyEvaluation(BaseModel):
    score: float = Field(description="Score between 0 and 1 indicating how relevant the answer is to the question.")
    reasoning: str = Field(description="Explanation for the relevancy score.")

# ============================================================================
# RAG EVALUATOR (LLM-as-judge)
# ============================================================================

class RAGEvaluator:
    """
    Implements core metrics from the RAGAS framework using Gemini 2.5 Flash as the judge.
    1. Faithfulness: Is the answer factually grounded in the context? (Hallucination check)
    2. Relevancy: Does the answer address the question?
    3. Context Quality: How useful were the retrieved chunks?
    """

    def __init__(self, model_name: str = "gemini-2.5-flash"):
        # temperature=0: evaluation must be repeatable and unbiased.
        self.llm = ChatGoogleGenerativeAI(model=model_name, temperature=0)
        self.parser_f = JsonOutputParser(pydantic_object=FaithfulnessEvaluation)
        self.parser_r = JsonOutputParser(pydantic_object=RelevancyEvaluation)

    def evaluate_faithfulness(self, question: str, answer: str, context: str) -> Dict[str, Any]:
        """Check for hallucinations. Higher score = More grounded in context."""
        prompt = ChatPromptTemplate.from_template(
            """Evaluate the faithfulness of the answer based on the context.
1. Extract the main claims from the answer.
2. For each claim, check if it can be verified using ONLY the provided context.

Question: {question}
Context: {context}
Answer: {answer}

{format_instructions}
"""
        )

        chain = prompt | self.llm | self.parser_f
        try:
            return chain.invoke({
                "question": question,
                "context": context,
                "answer": answer,
                "format_instructions": self.parser_f.get_format_instructions(),
            })
        except Exception as e:
            return {"overall_score": 0.0, "evaluations": [], "error": str(e)}

    def evaluate_relevancy(self, question: str, answer: str) -> Dict[str, Any]:
        """Check if the answer addresses the question directly."""
        prompt = ChatPromptTemplate.from_template(
            """Rate the relevancy of the answer to the question on a scale from 0.0 to 1.0.
0.0 means completely off-topic. 1.0 means perfectly addresses the query.

Question: {question}
Answer: {answer}

{format_instructions}
"""
        )

        chain = prompt | self.llm | self.parser_r
        try:
            return chain.invoke({
                "question": question,
                "answer": answer,
                "format_instructions": self.parser_r.get_format_instructions(),
            })
        except Exception as e:
            return {"score": 0.0, "reasoning": "Error occurred", "error": str(e)}

# ============================================================================
# RETRIEVAL METRICS (Classical)
# ============================================================================

class RetrievalMetrics:
    """Calculates non-LLM metrics for the retrieval component."""

    @staticmethod
    def calc_mrr(retrieved_ids: List[str], relevant_ids: List[str]) -> float:
        for i, doc_id in enumerate(retrieved_ids, 1):
            if doc_id in relevant_ids:
                return 1.0 / i
        return 0.0

    @staticmethod
    def calc_hit_rate(retrieved_ids: List[str], relevant_ids: List[str]) -> float:
        for doc_id in retrieved_ids:
            if doc_id in relevant_ids:
                return 1.0
        return 0.0

# ============================================================================
# DEMONSTRATION WORKFLOW
# ============================================================================

def run_task4_demo():
    print("=" * 60)
    print("🚀 RUNNING TASK 4: RAG Evaluation & LLM-as-judge")
    print("=" * 60)

    if not os.environ.get("GOOGLE_API_KEY"):
        print("⚠️ Set GOOGLE_API_KEY to run Task 4.")
        return None

    sample_q = "How effective are mRNA vaccines against different variants?"
    sample_context = """
    A study in NEJM showed that mRNA-1273 was 94% effective against early variants.
    However, efficacy dropped to 88% for the Delta variant. Omicron further reduced
    neutralizing activity but protection against severe disease remained high.
    """

    faithful_a = "mRNA vaccines show high overall efficacy (94%), though it decreases slightly for Delta (88%)."
    hallucinated_a = "mRNA vaccines are 100% effective against Omicron and prevent all transmission."

    evaluator = RAGEvaluator()

    print(f"\n1. Judging Faithfulness (Hallucination Detection)...")
    res_f = evaluator.evaluate_faithfulness(sample_q, faithful_a, sample_context)
    print(f"    Faithful Answer Score: {res_f['overall_score']}")

    res_h = evaluator.evaluate_faithfulness(sample_q, hallucinated_a, sample_context)
    print(f"    Hallucinated Answer Score: {res_h['overall_score']}")
    for eval_item in res_h['evaluations']:
        if not eval_item['supported_by_context']:
            print(f"      - Hallucination Detected: '{eval_item['statement']}'")
            print(f"        Reason: {eval_item['reasoning']}")

    print(f"\n2. Judging Answer Relevancy...")
    res_rel = evaluator.evaluate_relevancy(sample_q, faithful_a)
    print(f"   Relevancy Score: {res_rel['score']}")
    print(f"      Reasoning: {res_rel['reasoning']}")

    print(f"\n3. Classical Retrieval Metrics (MRR/Hit Rate)...")
    retrieved = ["doc_9", "doc_1", "doc_4"]
    relevant = ["doc_1", "doc_2"]

    mrr = RetrievalMetrics.calc_mrr(retrieved, relevant)
    hit = RetrievalMetrics.calc_hit_rate(retrieved, relevant)

    print(f"   - MRR: {mrr} (Relevant document was at Rank 2)")
    print(f"   - Hit Rate: {hit} (Found at least one relevant doc)")

    print("\n" + "=" * 60)
    print("TASK 4 COMPLETE")
    print("=" * 60)

if __name__ == "__main__":
    run_task4_demo()
